# 国会会議録 メタデータ分析
## Phase 1.5: メタデータ分析（本文取得前）

対象: 移民・外国人労働者 20キーワード, 2000-2026年
データ: 15,558 speeches (deduplicated)

**Note**: Run from `sou/` directory: `uv run jupyter lab`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

# Paths (relative to sou/ project root)
REPO_ROOT = Path(__file__).resolve().parents[2]   # goes up: notebooks -> sou -> repo root
DATA_DIR = REPO_ROOT / "sou" / "data" / "raw" / "kokkai"

# matplotlib: system CJK font
plt.rcParams['axes.unicode_minus'] = False

# Load metadata
META_FILE = DATA_DIR / "metadata.json"
with open(META_FILE, 'r', encoding='utf-8') as f:
    raw = json.load(f)

speeches = raw['speeches']
keywords = raw['metadata']['keywords']
print(f"Total speeches: {len(speeches)}")
print(f"Keywords: {len(keywords)}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Convert to DataFrame
records = []
for sid, s in speeches.items():
    records.append({
        'speechID': sid,
        'session': s.get('session'),
        'nameOfHouse': s.get('nameOfHouse'),
        'nameOfMeeting': s.get('nameOfMeeting'),
        'issue': s.get('issue'),
        'date': s.get('date'),
        'speaker': s.get('speaker'),
        'speakerGroup': s.get('speakerGroup'),
        'speakerPosition': s.get('speakerPosition'),
        'keywords_matched': s.get('keywords_matched', []),
        'num_keywords': len(s.get('keywords_matched', [])),
    })

df = pd.DataFrame(records)
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

print(f"DataFrame shape: {df.shape}")
df.head(3)

---
## 1️⃣ Keyword Frequency Distribution

In [ ]:
# Explode keywords to count per keyword
kw_exp = df.explode('keywords_matched')
kw_counts = kw_exp['keywords_matched'].value_counts().reset_index()
kw_counts.columns = ['keyword', 'unique_speeches']

print("=== Keyword Frequency ===")
print(kw_counts.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(kw_counts['keyword'], kw_counts['unique_speeches'], color='steelblue')
ax.set_xlabel('Unique Speeches')
ax.set_title('Keyword Frequency Distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig1_keyword_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2️⃣ Overlap Matrix (Keyword Co-occurrence)

In [ ]:
# Build speechID -> set of keywords
sid_to_kws = {sid: set(s.get('keywords_matched', [])) for sid, s in speeches.items()}

# All unique keywords
all_kws = []
for kws in sid_to_kws.values():
    all_kws.extend(kws)
unique_kws = sorted(list(set(all_kws)))
print(f"Unique keywords found: {len(unique_kws)}")

# Build co-occurrence matrix
n = len(unique_kws)
overlap_arr = np.zeros((n, n), dtype=int)
for i, kwi in enumerate(unique_kws):
    for j, kwj in enumerate(unique_kws):
        count = sum(1 for kws in sid_to_kws.values() if kwi in kws and kwj in kws)
        overlap_arr[i, j] = count

overlap = pd.DataFrame(overlap_arr, index=unique_kws, columns=unique_kws)
print("\n=== Overlap Matrix ===")
print(overlap.to_string())

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(overlap_arr, cmap='Blues', aspect='auto')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(unique_kws, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(unique_kws, fontsize=9)
plt.colorbar(im, ax=ax, label='Shared speeches')
ax.set_title('Keyword Overlap Matrix')
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig2_overlap_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top overlaps (off-diagonal)
top_overlaps = []
for i, kwi in enumerate(unique_kws):
    for j, kwj in enumerate(unique_kws):
        if i < j and overlap_arr[i, j] > 0:
            top_overlaps.append((kwi, kwj, int(overlap_arr[i, j])))
top_overlaps.sort(key=lambda x: -x[2])

print("\n=== Top 20 Keyword Pairs ===")
for kwi, kwj, c in top_overlaps[:20]:
    print(f"  {kwi} x {kwj}: {c}")

---
## 3️⃣ Time Distribution (Yearly)

In [ ]:
# Yearly speech count
yearly = df.groupby('year').size().reset_index(name='count')
print("=== Yearly Distribution ===")
print(yearly.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly['year'], yearly['count'], color='steelblue', alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Speech Count')
ax.set_title('Yearly Speech Count (2000-2026)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig3_yearly_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4️⃣ Party Distribution (speakerGroup)

In [ ]:
# Party distribution
party_counts = df['speakerGroup'].value_counts().reset_index()
party_counts.columns = ['party', 'count']
party_counts = party_counts.dropna()
print(f"=== Party Distribution (Top 30) ===")
print(party_counts.head(30).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 8))
top20 = party_counts.head(20)
ax.barh(top20['party'], top20['count'], color='coral')
ax.set_xlabel('Speech Count')
ax.set_title('Party Distribution (Top 20)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig4_party_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5️⃣ Committee / Meeting Distribution

In [ ]:
# Committee distribution
comm_counts = df['nameOfMeeting'].value_counts().reset_index()
comm_counts.columns = ['committee', 'count']
print("=== Committee Distribution (Top 30) ===")
print(comm_counts.head(30).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 8))
top20 = comm_counts.head(20)
ax.barh(top20['committee'], top20['count'], color='seagreen')
ax.set_xlabel('Speech Count')
ax.set_title('Committee Distribution (Top 20)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig5_committee_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary Stats

In [ ]:
print("=== Summary ===")
print(f"Total speeches: {len(df)}")
print(f"Date range: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"Sessions: {df['session'].nunique()}")
print(f"Unique speakers: {df['speaker'].nunique()}")
print(f"Parties: {df['speakerGroup'].nunique()}")
print(f"Committees: {df['nameOfMeeting'].nunique()}")
print(f"Speeches with 2+ keywords: {(df['num_keywords'] > 1).sum()}")